# Results for model: mistralai/mixtral-8x22b-instruct-v0.1

In [ ]:
import polars as pl
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import numpy as np

# Load data
df = pl.read_parquet('./jane_street_data/train.parquet', columns=['date_id', 'feature_00', 'responder_6'])

# Feature Engineering
df = df.sort(["date_id"])
top_quantile = df.select(pl.col("feature_00").rolling_quantile(0.15, window_size=30, min_periods=1, interpolate=True).alias("top_quantile"))
df = df.with_context(top_quantile).select(pl.exclude("top_quantile"), pl.col("top_quantile").fill_null(0.0).over("date_id"))

# Prepare data for training
X = df.drop("responder_6")
y = df["responder_6"]
X_train, X_test, y_train, y_test = train_test_split(X.to_numpy(), y.to_numpy(), test_size=0.2, random_state=42)

# Convert into DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# Train xgboost regressor
params = {"objective": "reg:squarederror"}
model = xgb.train(params=params, dtrain=dtrain, num_boost_round=100)

# Evaluate
y_pred = model.predict(dtest)
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error: {mae}")